# Base XGBoost Model

In [1]:
# import packages
import seaborn as sns
import xgboost
import torch
import numpy as np
import os
import gc
import pandas as pd
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import matplotlib.pyplot as plt
from time import time

In [2]:
# Use Cols defined by winner's EDA

# COLUMNS WITH STRINGS
str_type = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain','M1', 'M2', 'M3', 'M4','M5',
            'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 
            'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
str_type += ['id-12', 'id-15', 'id-16', 'id-23', 'id-27', 'id-28', 'id-29', 'id-30', 
            'id-31', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38']

# FIRST 53 COLUMNS
cols = ['TransactionID', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11',
       'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8',
       'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4',
       'M5', 'M6', 'M7', 'M8', 'M9']

# V COLUMNS TO LOAD DECIDED BY CORRELATION EDA
# https://www.kaggle.com/cdeotte/eda-for-columns-v-and-id
v =  [1, 3, 4, 6, 8, 11]
v += [13, 14, 17, 20, 23, 26, 27, 30]
v += [36, 37, 40, 41, 44, 47, 48]
v += [54, 56, 59, 62, 65, 67, 68, 70]
v += [76, 78, 80, 82, 86, 88, 89, 91]

#v += [96, 98, 99, 104] #relates to groups, no NAN 
v += [107, 108, 111, 115, 117, 120, 121, 123] # maybe group, no NAN
v += [124, 127, 129, 130, 136] # relates to groups, no NAN

# LOTS OF NAN BELOW
v += [138, 139, 142, 147, 156, 162] #b1
v += [165, 160, 166] #b1
v += [178, 176, 173, 182] #b2
v += [187, 203, 205, 207, 215] #b2
v += [169, 171, 175, 180, 185, 188, 198, 210, 209] #b2
v += [218, 223, 224, 226, 228, 229, 235] #b3
v += [240, 258, 257, 253, 252, 260, 261] #b3
v += [264, 266, 267, 274, 277] #b3
v += [220, 221, 234, 238, 250, 271] #b3

v += [294, 284, 285, 286, 291, 297] # relates to grous, no NAN
v += [303, 305, 307, 309, 310, 320] # relates to groups, no NAN
v += [281, 283, 289, 296, 301, 314] # relates to groups, no NAN
#v += [332, 325, 335, 338] # b4 lots NAN

cols += ['V'+str(x) for x in v]
dtypes = {}
for c in cols+['id_0'+str(x) for x in range(1,10)]+['id_'+str(x) for x in range(10,34)]+\
    ['id-0'+str(x) for x in range(1,10)]+['id-'+str(x) for x in range(10,34)]:
        dtypes[c] = 'float32'
for c in str_type: dtypes[c] = 'category'

In [3]:
# LOAD TRAIN
X_train = pd.read_csv('../input/ieee-fraud-detection/train_transaction.csv',index_col='TransactionID', dtype=dtypes, usecols=cols+['isFraud'])
train_id = pd.read_csv('../input/ieee-fraud-detection/train_identity.csv',index_col='TransactionID', dtype=dtypes)
X_train = X_train.merge(train_id, how='left', left_index=True, right_index=True)
# LOAD TEST
X_test = pd.read_csv('../input/ieee-fraud-detection/test_transaction.csv',index_col='TransactionID', dtype=dtypes, usecols=cols)
test_id = pd.read_csv('../input/ieee-fraud-detection/test_identity.csv',index_col='TransactionID', dtype=dtypes)
fix = {o:n for o, n in zip(test_id.columns, train_id.columns)}
test_id.rename(columns=fix, inplace=True)
X_test = X_test.merge(test_id, how='left', left_index=True, right_index=True)
# TARGET
y_train = X_train['isFraud'].copy()
del train_id, test_id, X_train['isFraud']; x = gc.collect()
# PRINT STATUS
print('Train shape',X_train.shape,'test shape',X_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../input/ieee-fraud-detection/train_transaction.csv'

In [ ]:
# Preprocess data
# NORMALIZE D COLUMNS
for i in range(1,16):
    if i in [1,2,3,5,9]: continue
    X_train['D'+str(i)] =  X_train['D'+str(i)] - X_train.TransactionDT/np.float32(24*60*60)
    X_test['D'+str(i)] = X_test['D'+str(i)] - X_test.TransactionDT/np.float32(24*60*60) 
# Convert M columns to Binary Values
for col in ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']:
    X_train[col] = X_train[col].map({'T': 1, 'F': 0, True: 1, False: 0}).astype('float32')
    X_test[col] = X_test[col].map({'T': 1, 'F': 0, True: 1, False: 0}).astype('float32')

In [ ]:
# FREQUENCY ENCODE TOGETHER
def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col],df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df1[nm] = df1[col].map(vc)
        df1[nm] = df1[nm].astype('float32')
        df2[nm] = df2[col].map(vc)
        df2[nm] = df2[nm].astype('float32')
        print(nm,', ',end='')
        
# LABEL ENCODE
def encode_LE(col,train=X_train,test=X_test,verbose=True):
    df_comb = pd.concat([train[col],test[col]],axis=0)
    df_comb,_ = df_comb.factorize(sort=True)
    nm = col
    if df_comb.max()>32000: 
        train[nm] = df_comb[:len(train)].astype('int32')
        test[nm] = df_comb[len(train):].astype('int32')
    else:
        train[nm] = df_comb[:len(train)].astype('int16')
        test[nm] = df_comb[len(train):].astype('int16')
    del df_comb; x=gc.collect()
    if verbose: print(nm,', ',end='')
        
# GROUP AGGREGATION MEAN AND STD
# https://www.kaggle.com/kyakovlev/ieee-fe-with-some-eda
def encode_AG(main_columns, uids, aggregations=['mean'], train_df=X_train, test_df=X_test, 
              fillna=True, usena=False):
    # AGGREGATION OF MAIN WITH UID FOR GIVEN STATISTICS
    for main_column in main_columns:
        print("encoding column: ",main_column)
        for col in uids:
            for agg_type in aggregations:
                new_col_name = main_column+'_'+col+'_'+agg_type
                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col,main_column]]])
                if usena: temp_df.loc[temp_df[main_column]==-1,main_column] = np.nan
                temp_df = temp_df.groupby([col])[main_column].agg([agg_type]).reset_index().rename(
                                                        columns={agg_type: new_col_name})

                temp_df.index = list(temp_df[col])
                temp_df = temp_df[new_col_name].to_dict()   

                train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')
                test_df[new_col_name]  = test_df[col].map(temp_df).astype('float32')
                
                if fillna:
                    # train_df[new_col_name].fillna(-1,inplace=True)
                    # test_df[new_col_name].fillna(-1,inplace=True)
                    train_df[new_col_name] = train_df[new_col_name].fillna(-1)
                    test_df[new_col_name] = test_df[new_col_name].fillna(-1)
                
                print("'"+new_col_name+"'",', ',end='')
                
# COMBINE FEATURES
def encode_CB(col1,col2,df1=X_train,df2=X_test):
    nm = col1+'_'+col2
    df1[nm] = df1[col1].astype(str)+'_'+df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str)+'_'+df2[col2].astype(str) 
    encode_LE(nm,verbose=False)
    print(nm,', ',end='')
    
# GROUP AGGREGATION NUNIQUE
def encode_AG2(main_columns, uids, train_df=X_train, test_df=X_test):
    for main_column in main_columns:  
        for col in uids:
            comb = pd.concat([train_df[[col]+[main_column]],test_df[[col]+[main_column]]],axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col+'_'+main_column+'_ct'] = test_df[col].map(mp).astype('float32')
            print(col+'_'+main_column+'_ct, ',end='')

In [ ]:
# TRANSACTION AMT CENTS
X_train['cents'] = (X_train['TransactionAmt'] - np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents'] = (X_test['TransactionAmt'] - np.floor(X_test['TransactionAmt'])).astype('float32')
print('cents, ', end='')
# FREQUENCY ENCODE: ADDR1, CARD1, CARD2, CARD3, P_EMAILDOMAIN
encode_FE(X_train,X_test,['addr1','card1','card2','card3','P_emaildomain'])
# COMBINE COLUMNS CARD1+ADDR1, CARD1+ADDR1+P_EMAILDOMAIN
encode_CB('card1','addr1')
encode_CB('card1_addr1','P_emaildomain')
# FREQUENCY ENOCDE
encode_FE(X_train,X_test,['card1_addr1','card1_addr1_P_emaildomain'])
# GROUP AGGREGATE
encode_AG(['TransactionAmt','D9','D11'],['card1','card1_addr1','card1_addr1_P_emaildomain'],['mean','std'],usena=True)

import datetime
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
X_train['DT_M'] = X_train['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_train['DT_M'] = (X_train['DT_M'].dt.year-2017)*12 + X_train['DT_M'].dt.month 

X_test['DT_M'] = X_test['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_test['DT_M'] = (X_test['DT_M'].dt.year-2017)*12 + X_test['DT_M'].dt.month

In [ ]:
# Add UID
X_train['day'] = X_train.TransactionDT / (24*60*60)
X_train['uid'] = X_train.card1_addr1.astype(str)+'_'+np.floor(X_train.day-X_train.D1).astype(str)

X_test['day'] = X_test.TransactionDT / (24*60*60)
X_test['uid'] = X_test.card1_addr1.astype(str)+'_'+np.floor(X_test.day-X_test.D1).astype(str)

In [ ]:
# FREQUENCY ENCODE UID
encode_FE(X_train,X_test,['uid'])
# AGGREGATE 
encode_AG(['TransactionAmt','D4','D9','D10','D15'],['uid'],['mean','std'],fillna=True,usena=True)
# AGGREGATE
encode_AG(['C'+str(x) for x in range(1,15) if x!=3],['uid'],['mean'],X_train,X_test,fillna=True,usena=True)
# AGGREGATE
encode_AG(['M'+str(x) for x in range(1,10)],['uid'],['mean'],fillna=True,usena=True)
# AGGREGATE
encode_AG2(['P_emaildomain','dist1','DT_M','id_02','cents'], ['uid'], train_df=X_train, test_df=X_test)
# AGGREGATE
encode_AG(['C14'],['uid'],['std'],X_train,X_test,fillna=True,usena=True)
# AGGREGATE 
encode_AG2(['C13','V314'], ['uid'], train_df=X_train, test_df=X_test)
# AGGREATE 
encode_AG2(['V127','V136','V309','V307','V320'], ['uid'], train_df=X_train, test_df=X_test)
# NEW FEATURE
X_train['outsider15'] = (np.abs(X_train.D1-X_train.D15)>3).astype('int8')
X_test['outsider15'] = (np.abs(X_test.D1-X_test.D15)>3).astype('int8')

In [ ]:
cols = list( X_train.columns )
cols.remove('TransactionDT')
for c in ['D6','D7','D8','D9','D12','D13','D14']:
    cols.remove(c)
for c in ['DT_M','day','uid']:
    cols.remove(c)
    
# FAILED TIME CONSISTENCY TEST
for c in ['C3','M5','id_08','id_33']:
    cols.remove(c)
for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    cols.remove(c)
for c in ['id_'+str(x) for x in range(22,28)]:
    cols.remove(c)

print('USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

In [ ]:
import joblib
import json
import os
from xgboost.callback import EarlyStopping

# Create directory for model artifacts
model_dir = '/kaggle/working/xgb'
os.makedirs(model_dir, exist_ok=True)

In [ ]:
oof = np.zeros(len(X_train))
preds = np.zeros(len(X_test))
fold_models = []  # Keep in memory for threshold tuning

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

def evaluate_fraud_model(y_true, probabilities, amounts, thresholds=None):
    """
    Evaluate fraud model with both count-based and dollar-weighted metrics.
    
    Parameters:
    - y_true: actual labels
    - probabilities: predicted probabilities
    - amounts: transaction amounts
    - thresholds: list of thresholds to evaluate (default: 0.1 to 0.9)
    """
    if thresholds is None:
        thresholds = np.arange(0.1, 0.95, 0.05)
    
    results = []
    
    total_fraud_count = y_true.sum()
    total_fraud_dollars = amounts[y_true == 1].sum()
    total_legit_count = (y_true == 0).sum()
    total_legit_dollars = amounts[y_true == 0].sum()
    
    for thresh in thresholds:
        preds = (probabilities >= thresh).astype(int)
        
        # Basic confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        
        # Count-based metrics
        fnr = fn / (fn + tp)  # false negative rate
        fpr = fp / (fp + tn)  # false positive rate
        recall = tp / (tp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Dollar-weighted metrics
        fraud_mask = y_true == 1
        legit_mask = y_true == 0
        
        missed_fraud_dollars = amounts[(fraud_mask) & (preds == 0)].sum()
        caught_fraud_dollars = amounts[(fraud_mask) & (preds == 1)].sum()
        blocked_legit_dollars = amounts[(legit_mask) & (preds == 1)].sum()
        
        dollar_weighted_fnr = missed_fraud_dollars / total_fraud_dollars
        dollar_weighted_recall = caught_fraud_dollars / total_fraud_dollars
        dollar_weighted_fpr = blocked_legit_dollars / total_legit_dollars
        
        results.append({
            'threshold': thresh,
            'recall': recall,
            'fnr': fnr,
            'precision': precision,
            'fpr': fpr,
            'missed_fraud_count': fn,
            'missed_fraud_dollars': missed_fraud_dollars,
            'dollar_weighted_recall': dollar_weighted_recall,
            'dollar_weighted_fnr': dollar_weighted_fnr,
            'blocked_legit_dollars': blocked_legit_dollars,
            'flagged_transactions': fp + tp
        })
    
    return pd.DataFrame(results)

def find_optimal_threshold(y_true, probabilities, amounts, min_recall=0.95, use_dollar_weighted=True):
    """
    Find threshold that achieves minimum recall while minimizing false positives.
    """
    results = evaluate_fraud_model(y_true, probabilities, amounts)
    
    recall_col = 'dollar_weighted_recall' if use_dollar_weighted else 'recall'
    
    # Filter to thresholds meeting recall constraint
    valid = results[results[recall_col] >= min_recall]
    
    if len(valid) == 0:
        print(f"Warning: No threshold achieves {min_recall:.0%} recall. Returning lowest threshold.")
        return results.iloc[0]['threshold'], results
    
    # Among valid thresholds, pick the one with lowest FPR (highest threshold)
    optimal = valid.loc[valid['fpr'].idxmin()]
    
    return optimal['threshold'], results

# Usage with your OOF predictions
results_df = evaluate_fraud_model(
    y_true=y_train,
    probabilities=oof,
    amounts=X_train['TransactionAmt']
)

In [ ]:
skf = GroupKFold(n_splits=6)
for i, (idxT, idxV) in enumerate(skf.split(X_train, y_train, groups=X_train['DT_M'])):
    month = X_train.iloc[idxV]['DT_M'].iloc[0]
    print('Fold', i, 'withholding month', month)
    print(' rows of train =', len(idxT), 'rows of holdout =', len(idxV))
    
    clf = xgb.XGBClassifier(
        n_estimators=5000,
        max_depth=12,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.4,
        missing=-1,
        eval_metric='auc',
        enable_categorical=True,
        callbacks=[EarlyStopping(rounds=100)],
        # nthread=4,
        # tree_method='hist'
        device='cuda'
        # tree_method='gpu_hist'
    )
    
    clf.fit(
        X_train[cols].iloc[idxT], y_train.iloc[idxT],
        eval_set=[(X_train[cols].iloc[idxV], y_train.iloc[idxV])],
        verbose=100
    )

    clf.set_params(predictor='cpu_predictor')
    
    oof[idxV] += clf.predict_proba(X_train[cols].iloc[idxV])[:, 1]
    preds += clf.predict_proba(X_test[cols])[:, 1] / skf.n_splits
    
    # Save each fold model
    model_path = f'{model_dir}/xgb_fold_{i}.joblib'
    joblib.dump(clf, model_path)
    print(f' Saved model to {model_path}')
    
    fold_models.append(clf)
    _ = gc.collect()

print('#' * 20)
print('XGB96 OOF CV=', roc_auc_score(y_train, oof))

# Find optimal threshold using your evaluation function
optimal_thresh, results_df = find_optimal_threshold(
    y_true=y_train,
    probabilities=oof,
    amounts=X_train['TransactionAmt'],
    min_recall=0.95,
    use_dollar_weighted=True
)

print(f'Optimal threshold: {optimal_thresh}')

# Save metadata needed for inference
metadata = {
    'feature_columns': list(cols),
    'n_folds': skf.n_splits,
    'threshold': float(optimal_thresh),
    'oof_auc': float(roc_auc_score(y_train, oof)),
    'model_params': {
        'n_estimators': 5000,
        'max_depth': 12,
        'learning_rate': 0.02,
        'subsample': 0.8,
        'colsample_bytree': 0.4,
        'missing': -1
    }
}

with open(f'{model_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Save OOF predictions for later analysis
np.save(f'{model_dir}/oof_predictions_xgb.npy', oof)
np.save(f'{model_dir}/test_predictions_xgb.npy', preds)
np.save(f'{model_dir}/y_train_xgb.npy', y_train.values)
np.save(f'{model_dir}/transaction_amounts_xgb.npy', X_train['TransactionAmt'].values)
np.save(f'{model_dir}/months_xgb.npy', X_train['DT_M'].values)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/xgb', 'zip', '/kaggle/working/xgb')